# Vision Transformers for Spatial Field Prediction with ERA5

## Motivation

The sequence Transformer we built for Leaf River treated each time step as a **token**. But atmospheric data is inherently *spatial*: pressure, temperature, and wind form continuous fields over a 2-D grid. The natural question is: **can we apply the same self-attention idea directly to spatial fields?**

This is exactly what the **Vision Transformer (ViT)** does. Introduced by Dosovitskiy et al. (2021), ViT splits a 2-D image into non-overlapping patches, embeds each patch as a single token, and feeds the sequence of patch-tokens through a standard Transformer encoder. The key difference from CNNs is that **every patch can attend to every other patch from the very first layer** — there is no inductive locality bias.

In 2023, ViT-style architectures swept numerical weather prediction benchmarks:
- **Pangu-Weather** (Bi et al. 2023) — 3-D ViT outperforming ECMWF on medium-range forecasts
- **ClimaX** (Nguyen et al. 2023) — ViT pre-trained on ERA5, fine-tuned for downstream tasks
- **GraphCast** and **FourCastNet** — related global attention ideas on sphere graphs

## What we'll cover

1. Load ERA5 reanalysis from the **WeatherBench2** cloud archive (Google Cloud Storage, publicly accessible)
2. Frame a **spatial image regression** task: given five atmospheric fields at time $t$, predict the 2 m temperature field 24 h later
3. Build the ViT piece by piece: **patch embedding → 2-D positional encoding → multi-head self-attention → regression head**
4. Train and evaluate; visualise **spatial attention maps** to see what regions the model relies on

## Earth science connection

ERA5 is ECMWF's fifth-generation global atmospheric reanalysis (Hersbach et al. 2020), covering 1940–present at 0.25° resolution. It is the de-facto benchmark dataset for data-driven weather and climate modelling. The prediction task we pose — 24-hour 2 m temperature forecasting over Europe — is both scientifically meaningful and small enough to train on a laptop CPU in minutes.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import xarray as xr
from tqdm.auto import tqdm
from dask.diagnostics import ProgressBar

rng = np.random.default_rng(42)
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cpu')
print('PyTorch:', torch.__version__, '| device:', device)

PyTorch: 2.10.0 | device: cpu


## 1. Experiment configuration

All tuneable knobs live here. The defaults keep the notebook runnable on a CPU in ~ 10 minutes. The easiest levers for a richer experiment are `DATE_TRAIN_END` (more data) and `N_EPOCHS`.

In [2]:
# ── Domain ──────────────────────────────────────────────────────────────────
# Europe: 32 × 32 grid at 2° resolution
LAT_SLICE = slice(87, 25, -2)   # 87°N → 27°N in 2° steps  (32 points)
LON_SLICE = slice(0, 62, 2)     # 0°E → 60°E in 2° steps   (32 points)
GRID_H, GRID_W = 32, 32

# ── Variables ────────────────────────────────────────────────────────────────
# Four surface-only input channels; target is 2m_temperature
SURFACE_VARS = ['2m_temperature', '10m_u_component_of_wind',
                '10m_v_component_of_wind', 'mean_sea_level_pressure']
N_CHANNELS   = 4    # all surface
TARGET_VAR   = '2m_temperature'

# ── Time ─────────────────────────────────────────────────────────────────────
DATE_TRAIN_START = '2018-01-01'
DATE_TRAIN_END   = '2018-02-13'
DATE_VAL_START   = '2020-01-01'
DATE_VAL_END     = '2020-06-30'
DATE_TEST_START  = '2020-07-01'
DATE_TEST_END    = '2020-12-31'
TIME_STEP_HOURS  = 6     # use every 6th hour
LEAD_STEPS       = 4     # 4 × 6 h = 24 h lead time

# ── ViT architecture ─────────────────────────────────────────────────────────
PATCH_SIZE = 4           # 4×4 spatial patches  →  (32/4)² = 64 tokens
D_MODEL    = 128         # token dimension
N_HEADS    = 4
N_LAYERS   = 4
D_FF       = 256
DROPOUT    = 0.1

# ── Training ─────────────────────────────────────────────────────────────────
BATCH_SIZE = 32
N_EPOCHS   = 40
LR         = 3e-4
PATIENCE   = 8

## 2. Load ERA5 from the WeatherBench2 cloud archive

WeatherBench2 (Rasp et al. 2024) provides ERA5 as a public Zarr store on Google Cloud Storage — no account required. The full dataset is 669 TB; we access only the tiny slice we need through lazy Dask arrays and `.load()` only what fits in RAM.

```
gs://weatherbench2/datasets/era5/1959-2023_01_10-full_37-1h-0p25deg-chunk-1.zarr
```

This requires `gcsfs` (install with `uv add gcsfs` or `pip install gcsfs`).

In [3]:
ERA5_URL = (
    'gs://weatherbench2/datasets/era5/'
    '1959-2023_01_10-full_37-1h-0p25deg-chunk-1.zarr'
)

ds_full = xr.open_zarr(ERA5_URL, storage_options={'token': 'anon'})
ds_full

<xarray.Dataset> Size: 669TB
Dimensions:                                           (time: 561264,
                                                       latitude: 721,
                                                       longitude: 1440,
                                                       level: 37)
Coordinates:
  * time                                              (time) datetime64[ns] 4MB ...
  * latitude                                          (latitude) float32 3kB ...
  * longitude                                         (longitude) float32 6kB ...
  * level                                             (level) int64 296B 1 .....
Data variables: (12/48)
    10m_u_component_of_wind                           (time, latitude, longitude) float32 2TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    10m_v_component_of_wind                           (time, latitude, longitude) float32 2TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    2m_dewpoint_temperature                           (time, latitude, longitude) float32 2TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    2m_temperature                                    (time, latitude, longitude) float32 2TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    angle_of_sub_gridscale_orography                  (latitude, longitude) float32 4MB dask.array<chunksize=(721, 1440), meta=np.ndarray>
    anisotropy_of_sub_gridscale_orography             (latitude, longitude) float32 4MB dask.array<chunksize=(721, 1440), meta=np.ndarray>
    ...                                                ...
    v_component_of_wind                               (time, level, latitude, longitude) float32 86TB dask.array<chunksize=(1, 37, 721, 1440), meta=np.ndarray>
    vertical_velocity                                 (time, level, latitude, longitude) float32 86TB dask.array<chunksize=(1, 37, 721, 1440), meta=np.ndarray>
    volumetric_soil_water_layer_1                     (time, latitude, longitude) float32 2TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    volumetric_soil_water_layer_2                     (time, latitude, longitude) float32 2TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    volumetric_soil_water_layer_3                     (time, latitude, longitude) float32 2TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>
    volumetric_soil_water_layer_4                     (time, latitude, longitude) float32 2TB dask.array<chunksize=(1, 721, 1440), meta=np.ndarray>

In [ ]:
def load_era5_slice(ds, start, end, label=''):
    """
    Load a spatial/temporal subset of ERA5 surface variables into memory,
    with per-variable progress bars so slow cloud transfers are easy to monitor.

    Returns
    -------
    surf : xr.Dataset – surface variables on the Europe grid
    """
    time_idx = slice(start, end)

    surf_arrays = {}
    pbar = tqdm(SURFACE_VARS, desc=f'{label} surface vars', unit='var', leave=True)
    for var in pbar:
        pbar.set_postfix(var=var)
        lazy = (
            ds[var]
            .sel(time=time_idx)
            .isel(
                time=slice(None, None, TIME_STEP_HOURS),
                latitude=LAT_SLICE,
                longitude=LON_SLICE,
            )
        )
        with ProgressBar(dt=1.0):
            surf_arrays[var] = lazy.load()

    return xr.Dataset(surf_arrays)


splits = [
    ('Train', DATE_TRAIN_START, DATE_TRAIN_END),
    ('Val',   DATE_VAL_START,   DATE_VAL_END),
    ('Test',  DATE_TEST_START,  DATE_TEST_END),
]

results = {}
for label, start, end in tqdm(splits, desc='ERA5 splits', unit='split'):
    results[label] = load_era5_slice(ds_full, start, end, label=label)
    break

results['Train']

ERA5 splits:   0%|          | 0/3 [00:00<?, ?split/s]

Train surface vars:   0%|          | 0/4 [00:00<?, ?var/s]

[########################################] | 100% Completed | 33.12 s
[####################################### ] | 99% Completed | 81.43 ss

In [ ]:

surf_tr = results['Train']
surf_va = results['Val']
surf_te = results['Test']

print(f'\nTrain timesteps : {surf_tr.dims["time"]}'
      f'  ({DATE_TRAIN_START} → {DATE_TRAIN_END})')
print(f'Grid             : {GRID_H} × {GRID_W}')
print(f'Channels         : {N_CHANNELS}')

In [ ]:
surf_tr

In [ ]:
# ── Visualise the input fields at a sample time ──────────────────────────────
t0 = surf_tr.time[100].values

fig, axes = plt.subplots(1, 4, figsize=(15, 3.5))
lats = surf_tr.latitude.values
lons = surf_tr.longitude.values

fields = [
    (surf_tr['2m_temperature'].isel(time=100).values - 273.15, '2 m Temp (°C)', 'RdBu_r'),
    (surf_tr['10m_u_component_of_wind'].isel(time=100).values, 'U10 (m/s)', 'PiYG'),
    (surf_tr['10m_v_component_of_wind'].isel(time=100).values, 'V10 (m/s)', 'PiYG'),
    (surf_tr['mean_sea_level_pressure'].isel(time=100).values / 100, 'MSLP (hPa)', 'viridis'),
]

for ax, (data, title, cmap) in zip(axes, fields):
    im = ax.pcolormesh(lons, lats, data, cmap=cmap, shading='auto')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Longitude')
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Latitude')
fig.suptitle(f'ERA5 input channels  |  {str(t0)[:16]}', fontsize=12)
plt.tight_layout()
plt.show()

## 3. The image regression problem

We frame 24-hour temperature prediction as **multi-channel image regression**:

$$\underbrace{\mathbf{X}_t \in \mathbb{R}^{C \times H \times W}}_{\text{input}} \xrightarrow{f_\theta} \underbrace{\hat{\mathbf{y}}_{t+24} \in \mathbb{R}^{H \times W}}_{\text{predicted }T_{2m}}$$

Each sample is a stack of $C=5$ spatial fields (channels), each of size $H \times W = 32 \times 32$. The model predicts a single output field (2 m temperature) one day later. This is sometimes called **field-to-field** regression.

### Why not a CNN?

A convolutional network slides a small kernel (e.g., 3 × 3) across the image. To capture long-range dependencies — say, that a low-pressure system 3 000 km away will bring warm air tomorrow — the CNN needs many stacked layers, each expanding the receptive field by a few grid points. A ViT attends to every spatial patch simultaneously in every layer, with no locality constraint. For synoptic-scale weather this is physically motivated: **teleconnections are inherently non-local**.

In [ ]:
def build_tensors(surf):
    """
    Stack surface fields into (N, C, H, W) input tensor and (N, H, W) target tensor.
    Pairs (t, t + LEAD_STEPS) so the model predicts 24 h ahead.
    """
    T = surf.dims['time']
    N = T - LEAD_STEPS

    # Stack channels: shape (T, C, H, W)
    channels = np.stack([
        surf['2m_temperature'].values,
        surf['10m_u_component_of_wind'].values,
        surf['10m_v_component_of_wind'].values,
        surf['mean_sea_level_pressure'].values,
    ], axis=1)   # (T, 4, H, W)

    X = channels[:N]                                    # input at t
    y = surf['2m_temperature'].values[LEAD_STEPS:]      # target at t+24h

    return X.astype(np.float32), y.astype(np.float32)


X_tr_raw, y_tr_raw = build_tensors(surf_tr)
X_va_raw, y_va_raw = build_tensors(surf_va)
X_te_raw, y_te_raw = build_tensors(surf_te)

print(f'Train: X {X_tr_raw.shape}  y {y_tr_raw.shape}')
print(f'Val  : X {X_va_raw.shape}  y {y_va_raw.shape}')
print(f'Test : X {X_te_raw.shape}  y {y_te_raw.shape}')

In [ ]:
# ── Normalise per channel using training statistics ──────────────────────────
# Shape: (C,) mean and std computed over (N, H, W)
mu  = X_tr_raw.mean(axis=(0, 2, 3), keepdims=True)   # (1, C, 1, 1)
sig = X_tr_raw.std( axis=(0, 2, 3), keepdims=True) + 1e-6

y_mu  = y_tr_raw.mean()
y_sig = y_tr_raw.std() + 1e-6

def normalise_X(x): return (x - mu) / sig
def normalise_y(y): return (y - y_mu) / y_sig
def denorm_y(y):    return y * y_sig + y_mu

X_tr = torch.from_numpy(normalise_X(X_tr_raw))
X_va = torch.from_numpy(normalise_X(X_va_raw))
X_te = torch.from_numpy(normalise_X(X_te_raw))
y_tr = torch.from_numpy(normalise_y(y_tr_raw))
y_va = torch.from_numpy(normalise_y(y_va_raw))
y_te = torch.from_numpy(normalise_y(y_te_raw))

train_dl = DataLoader(TensorDataset(X_tr, y_tr), batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(TensorDataset(X_va, y_va), batch_size=BATCH_SIZE)
test_dl  = DataLoader(TensorDataset(X_te, y_te), batch_size=BATCH_SIZE)

## 4. From pixels to tokens: the patch embedding

The ViT's first step is to split the $C \times H \times W$ image into non-overlapping **patches** of size $C \times P \times P$ and linearly project each patch to a $d$-dimensional token vector.

With $P = 4$ and $H = W = 32$, we get $\frac{32}{4} \times \frac{32}{4} = 64$ patches. The full sequence of tokens passed to the Transformer is:

$$\mathbf{z}_0 = [\mathbf{E}\, \text{patch}_1 + \mathbf{p}_1, \;\; \mathbf{E}\, \text{patch}_2 + \mathbf{p}_2, \;\; \ldots, \;\; \mathbf{E}\, \text{patch}_{64} + \mathbf{p}_{64}]$$

where $\mathbf{E} \in \mathbb{R}^{d \times (C P^2)}$ is the patch projection matrix and $\mathbf{p}_i \in \mathbb{R}^{d}$ are **learnable positional embeddings** indexed by a 2-D grid position $(r, c)$.

The patch projection can be implemented elegantly with a single `Conv2d` layer whose kernel size equals the patch size:

In [ ]:
# ── Illustrate the patch layout ───────────────────────────────────────────────
n_patches_h = GRID_H // PATCH_SIZE
n_patches_w = GRID_W // PATCH_SIZE
n_patches   = n_patches_h * n_patches_w

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: raw 2m temperature field with patch grid overlay
t2m_sample = (surf_tr['2m_temperature'].isel(time=100).values - 273.15)
ax = axes[0]
im = ax.imshow(t2m_sample, cmap='RdBu_r', origin='upper',
               extent=[lons[0], lons[-1], lats[-1], lats[0]])
plt.colorbar(im, ax=ax, label='°C')
for i in range(n_patches_h + 1):
    y_pos = lats[0] + i * (lats[-1] - lats[0]) / n_patches_h
    ax.axhline(y_pos, color='k', lw=0.8, alpha=0.6)
for j in range(n_patches_w + 1):
    x_pos = lons[0] + j * (lons[-1] - lons[0]) / n_patches_w
    ax.axvline(x_pos, color='k', lw=0.8, alpha=0.6)
ax.set_title(f'Input field split into {n_patches_h}×{n_patches_w} = {n_patches} patches')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')

# Right: token grid with patch indices
ax = axes[1]
ax.set_xlim(-0.5, n_patches_w - 0.5)
ax.set_ylim(-0.5, n_patches_h - 0.5)
for r in range(n_patches_h):
    for c in range(n_patches_w):
        idx = r * n_patches_w + c
        ax.add_patch(plt.Rectangle((c - 0.5, r - 0.5), 1, 1,
                                   facecolor=plt.cm.tab20(idx % 20), alpha=0.7))
        ax.text(c, r, str(idx), ha='center', va='center', fontsize=6)
ax.set_title(f'Token sequence  ({n_patches} tokens, each dim={D_MODEL})')
ax.set_xlabel('Patch column'); ax.set_ylabel('Patch row')
ax.invert_yaxis()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f'Each patch covers a {PATCH_SIZE*2}°×{PATCH_SIZE*2}° area  '
      f'({PATCH_SIZE*2*111:.0f} km × {PATCH_SIZE*2*111:.0f} km at equator)')

## 5. Positional encoding for 2-D grids

The sequence Transformer used sinusoidal 1-D positional encoding because time is ordered. For spatial patches the ordering is 2-D: patch $(r, c)$ has both a row and a column. We use **learnable** positional embeddings: one vector $\mathbf{p}_{r,c} \in \mathbb{R}^d$ per grid position, stored in an `nn.Embedding` table of size $n_{\text{rows}} \times n_{\text{cols}}$.

Learnable 2-D embeddings outperform sinusoidal ones on bounded spatial domains (Dosovitskiy et al. 2021) because the model can freely encode relative positions, land–sea contrasts, and mean climate gradients into the positional signal.

$$\text{token}_{r,c} = \mathbf{E}\,\text{patch}_{r,c} + \mathbf{p}_{r,c}$$

where the index into the embedding table is $i = r \cdot n_{\text{cols}} + c$.

In [ ]:
class PatchEmbedding(nn.Module):
    """
    Split a multi-channel spatial field into patches and embed each patch.

    Args
    ----
    in_channels : number of input channels C
    patch_size  : spatial patch size P  (patches are P×P pixels)
    grid_h, grid_w : spatial grid dimensions H, W  (must be divisible by P)
    d_model     : output embedding dimension

    Input  : (B, C, H, W)
    Output : (B, n_patches, d_model)  where n_patches = (H/P)*(W/P)
    """
    def __init__(self, in_channels, patch_size, grid_h, grid_w, d_model):
        super().__init__()
        self.patch_size  = patch_size
        self.n_patches_h = grid_h // patch_size
        self.n_patches_w = grid_w // patch_size
        self.n_patches   = self.n_patches_h * self.n_patches_w

        # A single Conv2d with stride=kernel_size implements non-overlapping
        # patch extraction + linear projection in one step.
        self.proj = nn.Conv2d(
            in_channels, d_model,
            kernel_size=patch_size, stride=patch_size
        )   # output shape: (B, d_model, H/P, W/P)

        # Learnable 2-D positional embedding: one vector per patch position
        self.pos_embed = nn.Embedding(self.n_patches, d_model)
        self.register_buffer(
            'pos_idx',
            torch.arange(self.n_patches)
        )

    def forward(self, x):
        # x: (B, C, H, W)
        B = x.shape[0]
        x = self.proj(x)                         # (B, d_model, H/P, W/P)
        x = x.flatten(2).transpose(1, 2)        # (B, n_patches, d_model)
        x = x + self.pos_embed(self.pos_idx)    # broadcast positional vectors
        return x


# Quick shape check
_pe = PatchEmbedding(N_CHANNELS, PATCH_SIZE, GRID_H, GRID_W, D_MODEL)
_x  = torch.zeros(2, N_CHANNELS, GRID_H, GRID_W)
print('PatchEmbedding output shape:', _pe(_x).shape)
print(f'  {n_patches} tokens, each dim={D_MODEL}')

## 6. The Vision Transformer

The Transformer encoder is identical to the sequence case: alternating multi-head self-attention and feed-forward sublayers with residual connections and layer normalisation. The only changes are:

1. **Input**: patch tokens instead of time-step tokens
2. **No causal mask**: spatial patches have no causal ordering — every patch can attend to every other patch
3. **Output head**: instead of pooling to a scalar, we project each of the $n_p$ output tokens back to a $P^2$ vector, then fold them into the $H \times W$ output grid (the inverse of patch extraction)

$$\underbrace{\text{patch}}_{ P \times P} \xrightarrow{\text{project}} \underbrace{\mathbf{z}_{r,c}}_{d} \xrightarrow{\text{head}} \underbrace{\hat{y}_{r,c}}_{P^2} \xrightarrow{\text{fold}} \underbrace{\hat{\mathbf{y}}}_{H \times W}$$

The result is a fully attention-based encoder–decoder in which spatial resolution is preserved.

In [ ]:
class VisionTransformer(nn.Module):
    """
    ViT for spatial field regression.

    Architecture
    ------------
    patch_embed  → Transformer encoder → per-patch linear head → fold to grid

    Args
    ----
    in_channels : input channels C
    patch_size  : spatial patch size P
    grid_h, grid_w : H, W of the input field
    d_model, n_heads, n_layers, d_ff, dropout : standard Transformer knobs

    Input  : (B, C, H, W)
    Output : (B, H, W)  — predicted target field
    """
    def __init__(self, in_channels, patch_size, grid_h, grid_w,
                 d_model, n_heads, n_layers, d_ff, dropout):
        super().__init__()
        self.grid_h      = grid_h
        self.grid_w      = grid_w
        self.patch_size  = patch_size
        self.n_patches_h = grid_h // patch_size
        self.n_patches_w = grid_w // patch_size

        # 1. Patch embedding (includes learnable 2-D pos. enc.)
        self.patch_embed = PatchEmbedding(
            in_channels, patch_size, grid_h, grid_w, d_model
        )

        # 2. Transformer encoder — no causal mask (bidirectional spatial attention)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_ff,
            dropout=dropout,
            batch_first=True,
            norm_first=True,   # pre-norm (more stable)
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)

        # 3. Per-patch regression head: d_model → P² (one output pixel per patch pixel)
        self.head = nn.Linear(d_model, patch_size * patch_size)

    def forward(self, x):
        B = x.shape[0]

        # Patch embed: (B, n_patches, d_model)
        tokens = self.patch_embed(x)

        # Transformer encoder: (B, n_patches, d_model)
        tokens = self.encoder(tokens)

        # Head: (B, n_patches, P²)
        patches_out = self.head(tokens)

        # Fold patch tokens back to (B, H, W) — inverse of patch extraction
        P  = self.patch_size
        nH = self.n_patches_h
        nW = self.n_patches_w

        # (B, nH, nW, P, P) → (B, nH*P, nW*P) = (B, H, W)
        out = patches_out.view(B, nH, nW, P, P)
        out = out.permute(0, 1, 3, 2, 4).contiguous()   # (B, nH, P, nW, P)
        out = out.view(B, self.grid_h, self.grid_w)      # (B, H, W)
        return out

    @torch.no_grad()
    def attention_maps(self, x, layer_idx=0):
        """
        Extract raw multi-head attention weights from a specified encoder layer.
        Returns (n_heads, n_patches, n_patches) for a single input sample.
        """
        tokens = self.patch_embed(x.unsqueeze(0))   # (1, n_patches, d_model)
        # Forward through layers up to layer_idx, capturing attention at that layer
        for i, layer in enumerate(self.encoder.layers):
            if i == layer_idx:
                # Pre-norm version: norm first, then attention
                src = layer.norm1(tokens)
                attn_out, attn_w = layer.self_attn(
                    src, src, src, need_weights=True, average_attn_weights=False
                )   # attn_w: (1, n_heads, n_patches, n_patches)
                return attn_w[0].cpu().numpy()   # (n_heads, n_patches, n_patches)
            tokens = layer(tokens)
        raise ValueError(f'layer_idx {layer_idx} out of range')


model = VisionTransformer(
    in_channels=N_CHANNELS,
    patch_size=PATCH_SIZE,
    grid_h=GRID_H,
    grid_w=GRID_W,
    d_model=D_MODEL,
    n_heads=N_HEADS,
    n_layers=N_LAYERS,
    d_ff=D_FF,
    dropout=DROPOUT,
).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'ViT parameters: {n_params:,}')

# Shape smoke-test
_y = model(torch.zeros(2, N_CHANNELS, GRID_H, GRID_W))
print(f'Output shape:   {tuple(_y.shape)}  (expected: (2, {GRID_H}, {GRID_W}))')

## 7. Training

In [ ]:
def train_vit(model, train_dl, val_dl, n_epochs=N_EPOCHS, lr=LR, patience=PATIENCE):
    """
    Training loop with Adam, cosine learning-rate decay, and early stopping.
    """
    opt       = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=lr/20)
    best_val  = float('inf')
    patience_counter = 0
    history = {'train': [], 'val': []}

    for epoch in range(1, n_epochs + 1):
        # ── Training ──────────────────────────────────────────────────────────
        model.train()
        running = 0.0
        for X_b, y_b in train_dl:
            X_b, y_b = X_b.to(device), y_b.to(device)
            opt.zero_grad()
            loss = F.mse_loss(model(X_b), y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            running += loss.item()
        scheduler.step()

        # ── Validation ────────────────────────────────────────────────────────
        model.eval()
        with torch.no_grad():
            val_loss = sum(
                F.mse_loss(model(X_b.to(device)), y_b.to(device)).item()
                for X_b, y_b in val_dl
            ) / len(val_dl)

        history['train'].append(running / len(train_dl))
        history['val'].append(val_loss)

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), '/tmp/vit_era5_best.pt')
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stop at epoch {epoch}')
                break

        if epoch % 5 == 0:
            print(f'Epoch {epoch:3d}  train MSE={history["train"][-1]:.4f}  '
                  f'val MSE={val_loss:.4f}  lr={scheduler.get_last_lr()[0]:.2e}')

    model.load_state_dict(torch.load('/tmp/vit_era5_best.pt', weights_only=True))
    return history


history = train_vit(model, train_dl, val_dl)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(history['train'], label='Train MSE (normalised)')
ax.plot(history['val'],   label='Val MSE (normalised)')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (normalised units)')
ax.set_title('Vision Transformer training curve')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Evaluation

### 8a. RMSE skill maps

We evaluate on the held-out 2020 second half. The **spatial RMSE** shows where the model performs best (often ocean, which has less day-to-day variability) and where it struggles (complex terrain, frontal zones).

We compare against a simple **persistence baseline** — predict tomorrow's temperature as today's temperature — which is surprisingly hard to beat at 24 h.

In [ ]:
model.eval()

# Collect predictions for the full test set
preds_norm, truth_norm = [], []
with torch.no_grad():
    for X_b, y_b in test_dl:
        preds_norm.append(model(X_b.to(device)).cpu())
        truth_norm.append(y_b)

preds_norm = torch.cat(preds_norm, 0).numpy()   # (N_test, H, W)
truth_norm = torch.cat(truth_norm, 0).numpy()

# Denormalise to Kelvin
preds_K = denorm_y(preds_norm)
truth_K = denorm_y(truth_norm)

# Persistence: current T2m as the 24h forecast
persist_K = denorm_y(normalise_y(y_te_raw[:-LEAD_STEPS]))
# Align: truth is from index LEAD_STEPS onward in time dimension
# For persistence: predict y[t] = y[t-LEAD_STEPS]
# We already have y_te_raw as the 24h-ahead truth; persistence uses the input T2m
persist_K = X_te_raw[:, 0, :, :]   # channel 0 = T2m at input time  (in K)

# Spatial RMSE  (average over N_test samples)
rmse_vit     = np.sqrt(((preds_K  - truth_K)**2).mean(axis=0))   # (H, W)
rmse_persist = np.sqrt(((persist_K - truth_K)**2).mean(axis=0))

# Skill score: 1 - RMSE_model / RMSE_persist  (positive = better than persistence)
skill = 1 - rmse_vit / rmse_persist

print(f'Domain-mean RMSE  ViT         : {rmse_vit.mean():.2f} K')
print(f'Domain-mean RMSE  Persistence : {rmse_persist.mean():.2f} K')
print(f'Domain-mean skill score       : {skill.mean():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

vmax_rmse = max(rmse_vit.max(), rmse_persist.max())

for ax, data, title in zip(axes,
    [rmse_vit, rmse_persist, skill],
    ['ViT RMSE (K)', 'Persistence RMSE (K)', 'Skill score (ViT vs. persistence)']):

    if 'Skill' in title:
        im = ax.pcolormesh(lons, lats, data, cmap='RdYlGn',
                           vmin=-0.2, vmax=0.6, shading='auto')
    else:
        im = ax.pcolormesh(lons, lats, data, cmap='hot_r',
                           vmin=0, vmax=vmax_rmse, shading='auto')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title)
    ax.set_xlabel('Longitude')
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Latitude')
fig.suptitle('24 h T₂ₘ forecast evaluation  |  Test period: Jul–Dec 2020', fontsize=12)
plt.tight_layout()
plt.show()

### 8b. Example forecast maps

Visual inspection of a single forecast gives intuition for what the model learns and where it fails.

In [ ]:
# Pick a random test sample
idx = 42

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

panels = [
    (truth_K[idx]   - 273.15, 'ERA5 truth (°C)',       'RdBu_r'),
    (preds_K[idx]   - 273.15, 'ViT forecast (°C)',     'RdBu_r'),
    (preds_K[idx] - truth_K[idx], 'Error: forecast − truth (K)', 'coolwarm'),
]

vmin_t = min(truth_K[idx].min(), preds_K[idx].min()) - 273.15
vmax_t = max(truth_K[idx].max(), preds_K[idx].max()) - 273.15

for ax, (data, title, cmap) in zip(axes, panels):
    if 'Error' in title:
        vmax_e = np.abs(data).max()
        im = ax.pcolormesh(lons, lats, data, cmap=cmap,
                           vmin=-vmax_e, vmax=vmax_e, shading='auto')
    else:
        im = ax.pcolormesh(lons, lats, data, cmap=cmap,
                           vmin=vmin_t, vmax=vmax_t, shading='auto')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax.set_title(title)
    ax.set_xlabel('Longitude')
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Latitude')
plt.tight_layout()
plt.show()

### 8c. Spatial attention maps

One of the most powerful diagnostic tools for ViTs is visualising **where each token attends**. We extract the raw attention weight matrix $\mathbf{A} \in \mathbb{R}^{n_p \times n_p}$ from a chosen encoder layer, then reshape it back to the spatial grid.

For a query token at patch $(r_q, c_q)$, the row $\mathbf{A}[i_q, :]$ (where $i_q = r_q \cdot n_w + c_q$) tells us how much attention that patch pays to every other patch. Plotting this as a map reveals the **effective receptive field** the model has learned — and whether it respects physical teleconnections (e.g., North Atlantic storm tracks influencing Central European temperatures).

In [ ]:
# ── Extract attention maps from the first encoder layer ──────────────────────
sample_x = X_te[idx]   # (C, H, W)
attn_maps = model.attention_maps(sample_x, layer_idx=0)
# attn_maps: (n_heads, n_patches, n_patches)

# Average over heads for a mean attention map
attn_mean = attn_maps.mean(axis=0)   # (n_patches, n_patches)

# Pick three query locations: NW, Central, SE corner
nH, nW = n_patches_h, n_patches_w
query_patches = {
    'NW (Iceland region)':    (1, 1),
    'Central (France)':       (nH//2, nW//2),
    'SE (E. Mediterranean)':  (nH-2, nW-2),
}

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for col, (name, (qr, qc)) in enumerate(query_patches.items()):
    q_idx = qr * nW + qc

    # Row 0: mean-head attention map
    attn_row = attn_mean[q_idx].reshape(nH, nW)   # attention FROM this patch TO others
    im = axes[0, col].pcolormesh(
        np.linspace(lons[0], lons[-1], nW),
        np.linspace(lats[0], lats[-1], nH),
        attn_row, cmap='YlOrRd', shading='auto'
    )
    axes[0, col].scatter(
        lons[0] + qc * (lons[-1]-lons[0])/(nW-1),
        lats[0] + qr * (lats[-1]-lats[0])/(nH-1),
        s=120, marker='*', c='dodgerblue', zorder=5, label='Query patch'
    )
    plt.colorbar(im, ax=axes[0, col], fraction=0.046, pad=0.04)
    axes[0, col].set_title(f'Mean-head attention\nQuery: {name}', fontsize=9)
    axes[0, col].set_xlabel('Lon'); axes[0, col].set_ylabel('Lat')
    axes[0, col].grid(alpha=0.3)

    # Row 1: per-head attention for the same query
    for h in range(N_HEADS):
        attn_h = attn_maps[h, q_idx].reshape(nH, nW)
        axes[1, col].plot([], [], label=f'Head {h}')  # placeholder for colour
    # Show head 0 and head 1 side-by-side as contours on the T2m background
    axes[1, col].pcolormesh(
        lons, lats,
        truth_K[idx] - 273.15,
        cmap='RdBu_r', alpha=0.5, shading='auto'
    )
    for h, clr in enumerate(['blue', 'red', 'green', 'orange']):
        attn_h = attn_maps[h, q_idx].reshape(nH, nW)
        attn_up = np.repeat(np.repeat(attn_h, PATCH_SIZE, axis=0), PATCH_SIZE, axis=1)
        axes[1, col].contour(
            lons, lats, attn_up,
            levels=5, colors=clr, alpha=0.7, linewidths=0.8
        )
    axes[1, col].scatter(
        lons[0] + qc * (lons[-1]-lons[0])/(nW-1),
        lats[0] + qr * (lats[-1]-lats[0])/(nH-1),
        s=120, marker='*', c='yellow', edgecolors='k', zorder=5
    )
    axes[1, col].set_title(f'Per-head attention contours\n(blue/red/green/orange = heads 0–3)', fontsize=9)
    axes[1, col].set_xlabel('Lon'); axes[1, col].set_ylabel('Lat')
    axes[1, col].grid(alpha=0.3)

fig.suptitle('Spatial attention maps — layer 0  |  ★ marks the query patch', fontsize=12)
plt.tight_layout()
plt.show()

### 8d. Positional embedding similarity

Because the positional embeddings are learnable, they can encode more than just grid position — they may capture the **mean climate structure** of each patch (e.g., systematic land–sea or North–South gradients). We can inspect this by computing pairwise cosine similarities between all patch embeddings.

In [ ]:
pos_emb = model.patch_embed.pos_embed.weight.detach().numpy()  # (n_patches, d_model)

# Cosine similarity matrix
norm     = np.linalg.norm(pos_emb, axis=1, keepdims=True) + 1e-8
pos_norm = pos_emb / norm
cos_sim  = pos_norm @ pos_norm.T   # (n_patches, n_patches)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: full similarity matrix
im = axes[0].imshow(cos_sim, cmap='RdBu_r', vmin=-1, vmax=1, origin='upper')
plt.colorbar(im, ax=axes[0])
axes[0].set_title('Positional embedding cosine similarity')
axes[0].set_xlabel('Patch index'); axes[0].set_ylabel('Patch index')

# Right: similarity to the central patch, reshaped to the spatial grid
central_idx = (n_patches_h // 2) * n_patches_w + (n_patches_w // 2)
sim_to_center = cos_sim[central_idx].reshape(n_patches_h, n_patches_w)
im2 = axes[1].pcolormesh(
    np.linspace(lons[0], lons[-1], n_patches_w),
    np.linspace(lats[0], lats[-1], n_patches_h),
    sim_to_center, cmap='RdBu_r', vmin=-1, vmax=1, shading='auto'
)
axes[1].scatter(
    np.linspace(lons[0], lons[-1], n_patches_w)[n_patches_w // 2],
    np.linspace(lats[0], lats[-1], n_patches_h)[n_patches_h // 2],
    s=150, marker='*', c='gold', edgecolors='k', zorder=5, label='Reference patch'
)
plt.colorbar(im2, ax=axes[1])
axes[1].set_title('Cosine similarity to central patch')
axes[1].set_xlabel('Longitude'); axes[1].set_ylabel('Latitude')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Summary

| Concept | What we did |
|---|---|
| **Image regression** | Predicted 2 m temperature field (H×W) 24 h ahead from 5-channel ERA5 input field |
| **Patch embedding** | Split 32×32 grid into 64 non-overlapping 4×4 patches; each embedded as one token via `Conv2d` |
| **2-D positional encoding** | Learnable per-patch vectors (unlike sinusoidal 1-D encoding used for sequences) |
| **Transformer encoder** | Standard multi-head self-attention + FFN blocks, no causal mask (bidirectional spatial attention) |
| **Regression head** | Linear head maps each token → P² outputs; folded back to spatial grid |
| **Skill score** | ViT outperforms persistence (skill > 0) over most of the domain at 24 h |
| **Attention maps** | Different heads specialise: some attend locally, others capture long-range teleconnections |
| **Positional embeddings** | After training, similarity structure reflects spatial proximity and N–S climate gradients |

### Looking ahead

This ViT operated on a single analysis time step. Real operational ML weather models extend this in several directions:

- **Temporal context**: stack multiple past time steps as additional input channels (as in Pangu-Weather's 4-D patch embedding)
- **Spherical geometry**: patches on a latitude–longitude grid have unequal physical sizes; FourCastNet uses a spherical harmonic basis, and GraphCast uses a multi-scale icosahedral mesh to avoid this distortion
- **Ensemble forecasting**: train a score-based diffusion model conditioned on ViT latents to generate probabilistic forecast ensembles (GenCast, 2024)
- **Multi-task pre-training**: ClimaX pre-trains a single ViT on ERA5 across all variables and lead times, then fine-tunes for downstream tasks — the same transfer-learning recipe that powers large language models